<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #1e3a8a; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Taller Hands-On: El Agua de un Acueducto Veredal que Llega Sucia 💧
      </h1>
      <p style="margin: 6px 0 0 0; color: #1e3a8a; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #1e3a8a; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Taller Práctico Evaluativo · Módulo 05
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #2563eb; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Estadistica%20analisis%20y%20representacion%20de%20datos/homeworks/05_Exploracion_Limpieza_Hands_On.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## Objetivos de Aprendizaje 🔎

Este taller consolida el **Módulo 05: Exploración y Limpieza de Datos** ([README del módulo](../05%20-%20Exploracion%20y%20Limpieza%20de%20Datos/README.md)) con un problema **nuevo**: la calidad del agua de un **acueducto veredal ficticio** (sensores de turbidez, análisis de laboratorio, registros manuales de campo y una red de sensores fisicoquímicos).

Al terminar serás capaz de:

1. **Explorar con rigor:** resumir una variable (posición, escala, forma e IC *t*), separar un **error de captura** de un valor extremo legítimo, elegir la transformación de **Box-Cox** con su intervalo de verosimilitud y medir cuántos «hallazgos» **falsos** produce mirar muchas parejas de variables.
2. **Diagnosticar y tratar datos faltantes:** distinguir MCAR, MAR y MNAR con lo observado, **cuantificar el sesgo** de borrar o imputar con una verdad conocida y combinar imputaciones con las **reglas de Rubin**.
3. **Limpiar inconsistencias sin perder filas:** normalizar texto, retirar **centinelas**, convertir unidades, leer fechas con **formato explícito**, separar duplicados y llevar un **recibo de conservación** con cuarentena.
4. **Reducir dimensionalidad sin engañarte:** implementar **PCA con la SVD** (contrastado con `scikit-learn`), decidir el número de componentes (varianza acumulada, Kaiser y **análisis paralelo**), verificar la **identidad del error de reconstrucción** y detectar una **fuga de datos** en la selección de variables.

> 📚 **Repasa si lo necesitas:** [Resumen y EDA](../05%20-%20Exploracion%20y%20Limpieza%20de%20Datos/00_Introduccion_Cap2_Resumen_y_EDA.ipynb) · [Datos faltantes e inconsistentes](../05%20-%20Exploracion%20y%20Limpieza%20de%20Datos/01_Datos_Faltantes_e_Inconsistentes.ipynb) · [Selección y reducción de dimensionalidad](../05%20-%20Exploracion%20y%20Limpieza%20de%20Datos/02_Seleccion_y_Reduccion_de_Dimensionalidad_PCA_tSNE.ipynb).

---
## Instrucciones 📋

1. **Resuelve de forma autónoma:** en cada reto completa la celda de código marcada con `# TODO:`. Este cuaderno **no trae soluciones**.
2. **Autoverificación:** debajo de cada reto hay una celda con `assert`. Si tu solución es correcta imprime ✅; si no, el mensaje te dice qué revisar. No la modifiques.
3. **Puntaje:** cada reto indica su puntaje sugerido; el total del taller suma **100 puntos**.
4. Antes de entregar usa *Restart & Run All*: el cuaderno debe correr de principio a fin, sin errores y **sin `# TODO` pendientes**.
5. Todo funciona **sin internet y sin archivos externos** (datos sintéticos con semilla fija; el acueducto y sus cifras son **inventados**). Bibliotecas: `numpy`, `pandas`, `scipy` y `scikit-learn`, más `difflib` y `unicodedata` de la biblioteca estándar. Todo el cuaderno corre en pocos segundos.

In [ ]:
# ============================================================
# Preparación (no modificar): imports y datos sintéticos
# ============================================================
import difflib
import math
import unicodedata

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_predict, cross_val_score
from sklearn.pipeline import make_pipeline

SEMILLA = 2026


# --- Reto 1: turbidez (NTU) de un sensor con errores de captura, y 20 variables de puro ruido ---
def generar_turbidez(semilla=SEMILLA, n=500):
    """Turbidez lognormal (asimétrica) con 2 lecturas imposibles (error del sensor) en posiciones al azar."""
    rng = np.random.default_rng(semilla)
    t = np.round(rng.lognormal(mean=1.2, sigma=0.8, size=n), 2)
    t[rng.choice(n, 2, replace=False)] = [5000.0, 7500.0]
    return t


def generar_ruido(semilla=SEMILLA, n=120, p=20):
    """Matriz n x p de ruido gaussiano independiente: NO hay ninguna relación real entre columnas."""
    return np.random.default_rng(semilla + 1).normal(size=(n, p))


# --- Reto 2: análisis de agua completos (verdad conocida) y mecanismos de falta sobre `nitratos` ---
def generar_completos(semilla=SEMILLA, n=1000):
    rng = np.random.default_rng(semilla + 2)
    z = rng.multivariate_normal([0, 0], [[1, 0.5], [0.5, 1]], n)
    cond, dur = 180 + 40 * z[:, 0], 90 + 15 * z[:, 1]
    nit = 6 + 0.05 * (cond - 180) + 0.12 * (dur - 90) + rng.normal(0, 1.5, n)
    return pd.DataFrame({"conductividad": cond.round(1), "dureza": dur.round(1), "nitratos": nit.round(2)})


def mascara(df, mecanismo, rng, frac=0.3):
    """True = `nitratos` se pierde. MCAR: al azar | MAR: depende de la conductividad (observada) | MNAR: depende de los propios nitratos."""
    n = len(df)
    if mecanismo == "MCAR":
        s = rng.random(n)
    elif mecanismo == "MAR":
        s = (df["conductividad"] - df["conductividad"].mean()) / df["conductividad"].std() + rng.normal(0, 0.5, n)
    else:
        s = (df["nitratos"] - df["nitratos"].mean()) / df["nitratos"].std() + rng.normal(0, 0.5, n)
    return pd.Series(np.asarray(s) >= np.quantile(s, 1 - frac), index=df.index)


def imputar_estocastica(df, falta, rng):
    """Una imputación estocástica: regresión OLS sobre lo observado + ruido normal con la desviación residual."""
    obs, X, y = ~falta.to_numpy(), df[["conductividad", "dureza"]].to_numpy(), df["nitratos"].to_numpy()
    lr = LinearRegression().fit(X[obs], y[obs])
    s = (y[obs] - lr.predict(X[obs])).std(ddof=3)
    out = y.copy()
    out[~obs] = lr.predict(X[~obs]) + rng.normal(0, s, int((~obs).sum()))
    return out


datos = generar_completos()
MECANISMOS = ["MCAR", "MAR", "MNAR"]
faltas = {m: mascara(datos, m, np.random.default_rng(SEMILLA + 10 + i)) for i, m in enumerate(MECANISMOS)}


# --- Reto 3: registros manuales de campo (sucios) ---
VEREDAS = ["Alto de la Cruz", "El Rosal", "La Esperanza", "San Isidro", "Sabaneta", "Quebrada Honda", "Cañada Larga", "Pozo Azul"]
DESCONOCIDAS = ["Las Palmeras", "Monte Frío", "Villa Nueva", "El Mirador"]   # NO están en el catálogo


def _sin_tilde(t):
    return "".join(c for c in unicodedata.normalize("NFKD", t) if not unicodedata.combining(c))


def generar_registros(semilla=SEMILLA, n=240):
    """Devuelve (registros_sucios, verdad). Defectos sembrados: escritura de la vereda, centinelas, mL vs L, fechas, duplicados."""
    rng = np.random.default_rng(semilla + 3)
    canon = [str(v) for v in rng.choice(VEREDAS, n)]
    desc = set(int(i) for i in rng.choice(n, 8, replace=False))            # 8 veredas fuera del catálogo
    vereda = []
    for i, c in enumerate(canon):
        base = str(rng.choice(DESCONOCIDAS)) if i in desc else c
        k = int(rng.integers(0, 5))
        if k == 1:
            base = base.upper()
        elif k == 2:
            base = "  " + base.lower().replace(" ", "   ") + " "
        elif k == 3:
            base = _sin_tilde(base)
        elif k == 4 and i not in desc:
            j = int(rng.integers(1, len(base) - 1))
            base = base[:j] + base[j + 1:]                                  # errata: falta una letra
        vereda.append(base)
    dia, mes = rng.integers(1, 29, n), rng.integers(1, 13, n)
    fecha = [f"{d:02d}/{m:02d}/2024" for d, m in zip(dia, mes)]             # SIEMPRE día/mes/año
    turb = np.round(rng.lognormal(1.2, 0.8, n), 2)
    cent = rng.choice(n, 10, replace=False)
    turb[cent[:6]] = -999.0                                                 # centinela de «sin lectura»
    turb[cent[6:]] = 9999.0                                                 # centinela de «sensor saturado»
    vol_l = np.round(rng.uniform(0.5, 5.0, n), 3)
    en_ml = rng.random(n) < 0.35
    df = pd.DataFrame({"id_muestra": [f"M-{i + 1:03d}" for i in range(n)], "vereda": vereda, "fecha": fecha,
                       "turbidez_ntu": turb, "volumen": np.where(en_ml, vol_l * 1000, vol_l), "unidad": np.where(en_ml, "mL", "L")})
    verdad = pd.DataFrame({"vereda": [np.nan if i in desc else c for i, c in enumerate(canon)], "volumen_l": vol_l,
                           "fecha": pd.to_datetime(fecha, format="%d/%m/%Y")})
    dup = rng.choice(n, 10, replace=False)                                  # 10 filas copiadas idénticas
    df = pd.concat([df, df.iloc[dup]]); verdad = pd.concat([verdad, verdad.iloc[dup]])
    orden = rng.permutation(len(df))
    return df.iloc[orden].reset_index(drop=True), verdad.iloc[orden].reset_index(drop=True)


encuesta, _VERDAD = generar_registros()


# --- Reto 4: 10 sensores fisicoquímicos con escalas muy distintas, movidos por 3 factores latentes ---
COLUMNAS = ["conductividad", "ph", "dureza", "oxigeno", "alcalinidad", "turbidez", "nitratos", "solidos", "cloro", "temperatura"]
N_FACTORES = 3


def generar_sensores(semilla=SEMILLA, n=300):
    rng = np.random.default_rng(semilla + 4)
    F = rng.normal(size=(n, N_FACTORES))
    L = np.array([[.9, .1, 0], [.85, .2, 0], [.8, 0, .1], [0, .9, .1], [.1, .85, 0],
                  [0, .8, .2], [0, .1, .9], [.1, 0, .85], [.2, 0, .8], [.3, .3, .3]])
    Z = F @ L.T + rng.normal(0, 0.45, (n, len(COLUMNAS)))
    escala = np.array([300, 1, 20, 1, 50, 0.5, 5, 100, 2, 10.0])
    centro = np.array([200, 7, 30, 5, 60, 1, 10, 150, 3, 20.0])
    return Z * escala + centro


X_sens = generar_sensores()
print(f"Turbidez: {len(generar_turbidez())} lecturas | Análisis completos: {len(datos)} filas | Registros de campo: {len(encuesta)} filas | Sensores: {X_sens.shape}")

---
## Reto 1: Mirar antes de calcular — turbidez, Box-Cox y falsos hallazgos 🔍

**Contexto:** el sensor de turbidez del acueducto entrega `turbidez` en NTU (unidades nefelométricas). La variable es **positiva y muy asimétrica**, y el sensor a veces entrega lecturas **físicamente imposibles** (el equipo no mide más de `LIMITE_SENSOR = 1000` NTU). Además, un analista «explora» 20 variables de **puro ruido** y quiere saber cuántas correlaciones «significativas» saldrían.

**Enunciado**
1. `turb_cruda = generar_turbidez()`. Calcula `n_fuera` (lecturas por encima de `LIMITE_SENSOR`) y `turb` (el arreglo **sin** esas lecturas).
2. `resumen(x)` → diccionario con `n`, `media`, `mediana`, `sd` (muestral, $n-1$), `asimetria` (coeficiente de Fisher-Pearson, $g_1 = m_3/m_2^{3/2}$), `curtosis` (**exceso**, $m_4/m_2^2-3$) e `ic95` (tupla, IC $t$ de la media). Obtén `r_cruda = resumen(turb_cruda)` y `r_ok = resumen(turb)`, y `desplazamiento = r_cruda["media"] - r_ok["media"]`.
3. `tukey(x)` → número de puntos fuera de las vallas $[Q_1-1.5\,\text{RIC},\,Q_3+1.5\,\text{RIC}]$. Calcula `n_at_turb = tukey(turb)` y `n_at_log = tukey(np.log(turb))`.
4. **Box-Cox:** implementa `llf(lam, x)`, la log-verosimilitud perfilada $\ell(\lambda)=-\frac{n}{2}\ln\hat\sigma^2_\lambda+(\lambda-1)\sum_i\ln x_i$, donde $\hat\sigma^2_\lambda$ es la varianza (con divisor $n$) de $y_i=(x_i^\lambda-1)/\lambda$ (y $y_i=\ln x_i$ si $\lambda=0$). Sobre `grilla = np.linspace(-1, 2, 301)` obtén `lam_hat` (el máximo) e `ic_lambda = (inf, sup)`: los $\lambda$ de la grilla con $\ell(\lambda)\ge\ell(\hat\lambda)-\tfrac12\chi^2_{1,0.95}$.
5. **Comparaciones múltiples:** `ruido = generar_ruido()` (120 × 20). Calcula el valor p de la correlación de Pearson de **cada pareja** $i<j$ (`p_vals`, 190 valores), `n_raw` (cuántos con $p<0.05$) y `n_bonf` (cuántos con $p<0.05/190$).

**Criterios de aceptación:** `resumen` coincide con SciPy; tu `llf` coincide con `scipy.stats.boxcox_llf` y tu `lam_hat` con el de `scipy.stats.boxcox`; el IC de $\lambda$ **excluye** el 1 (no transformar); la regla de Tukey marca **menos** puntos sobre $\ln x$ que sobre $x$; y `n_bonf <= n_raw`.

**Puntaje sugerido:** 20 puntos.

In [ ]:
LIMITE_SENSOR = 1000.0
turb_cruda = generar_turbidez()

# TODO: completa
n_fuera = None          # lecturas por encima de LIMITE_SENSOR
turb = None             # turb_cruda sin esas lecturas


def resumen(x):
    # TODO: {"n", "media", "mediana", "sd", "asimetria", "curtosis", "ic95"}
    return None


r_cruda = r_ok = desplazamiento = None


def tukey(x):
    # TODO: número de puntos fuera de [Q1 - 1.5 RIC, Q3 + 1.5 RIC]
    return None


n_at_turb = n_at_log = None


def llf(lam, x):
    # TODO: log-verosimilitud perfilada de Box-Cox (varianza con divisor n)
    return None


grilla = np.linspace(-1, 2, 301)
lam_hat = None          # el lambda de la grilla que maximiza llf
ic_lambda = None        # (inf, sup) con llf >= max - 0.5 * chi2_{1, 0.95}

ruido = generar_ruido()
p_vals = None           # 190 valores p de Pearson (una por pareja i < j)
n_raw = n_bonf = None

In [ ]:
# Autoverificación del Reto 1 (no modificar)
assert n_fuera == 2 and turb is not None and len(turb) == len(turb_cruda) - 2 and turb.max() <= LIMITE_SENSOR, "n_fuera cuenta las lecturas > LIMITE_SENSOR (hay 2 sembradas) y turb es el resto."
_r = resumen(turb)
assert _r is not None and _r["n"] == len(turb), "resumen debe devolver un diccionario con la clave 'n'."
assert abs(_r["media"] - np.mean(turb)) < 1e-9 and abs(_r["mediana"] - np.median(turb)) < 1e-9 and abs(_r["sd"] - np.std(turb, ddof=1)) < 1e-9, "media, mediana y sd (con ddof=1) deben coincidir con NumPy."
assert abs(_r["asimetria"] - stats.skew(turb)) < 1e-9 and abs(_r["curtosis"] - stats.kurtosis(turb)) < 1e-9, "asimetria = m3 / m2**1.5 y curtosis (exceso) = m4 / m2**2 - 3, con momentos centrales de divisor n."
_lo, _hi = stats.t.interval(0.95, len(turb) - 1, loc=np.mean(turb), scale=stats.sem(turb))
assert abs(_r["ic95"][0] - _lo) < 1e-9 and abs(_r["ic95"][1] - _hi) < 1e-9, "ic95 = media ± t_{0.975, n-1} · sd / sqrt(n)."
assert r_cruda["n"] == len(turb_cruda) and r_ok["n"] == len(turb) and abs(desplazamiento - (np.mean(turb_cruda) - np.mean(turb))) < 1e-9 and desplazamiento > 10, "desplazamiento = media con errores - media sin errores (debe ser grande: dos lecturas imposibles arrastran la media)."
assert r_ok["asimetria"] > 1, "Sin los errores, la turbidez sigue siendo muy asimétrica a la derecha (g1 > 1)."


def _tk(v):
    q1, q3 = np.percentile(v, [25, 75]); r = q3 - q1
    return int(np.sum((v < q1 - 1.5 * r) | (v > q3 + 1.5 * r)))


assert n_at_turb == _tk(turb) and n_at_log == _tk(np.log(turb)), "tukey usa np.percentile (25 y 75) y cuenta los puntos fuera de las vallas."
assert n_at_log < n_at_turb, "Sobre la escala logarítmica Tukey debe marcar menos puntos: en una variable asimétrica las «vallas» sobre la escala original marcan la cola natural."
for _l in (-0.5, 0.0, 0.3, 1.0):
    assert abs(llf(_l, turb) - stats.boxcox_llf(_l, turb)) < 1e-8, f"llf({_l}) no coincide con scipy.stats.boxcox_llf: revisa el término (lam - 1) * sum(log x) y el divisor n de la varianza."
assert abs(lam_hat - stats.boxcox(turb)[1]) < 0.02, "lam_hat debe ser el lambda de la grilla con mayor llf (cerca del que da scipy.stats.boxcox)."
assert ic_lambda is not None and ic_lambda[0] <= lam_hat <= ic_lambda[1] < 1, "El IC de lambda debe contener a lam_hat y, por la fuerte asimetría, NO contener a 1 (no transformar)."
_c = stats.chi2.ppf(0.95, 1) / 2
_ok = [l for l in grilla if llf(l, turb) >= llf(lam_hat, turb) - _c]
assert abs(ic_lambda[0] - min(_ok)) < 1e-9 and abs(ic_lambda[1] - max(_ok)) < 1e-9, "ic_lambda = (mín, máx) de los lambda de la grilla con llf >= llf(lam_hat) - chi2.ppf(0.95, 1) / 2."
_iu = np.triu_indices(20, 1)
_p = np.array([stats.pearsonr(ruido[:, i], ruido[:, j])[1] for i, j in zip(*_iu)])
assert p_vals is not None and len(p_vals) == 190 and np.allclose(np.sort(np.asarray(p_vals)), np.sort(_p)), "p_vals: un valor p de Pearson por pareja i < j (20 · 19 / 2 = 190)."
assert n_raw == int((_p < 0.05).sum()) and n_bonf == int((_p < 0.05 / 190).sum()), "n_raw cuenta p < 0.05 y n_bonf cuenta p < 0.05 / 190."
assert n_raw >= 1 and n_bonf <= n_raw, "En 190 parejas de ruido deberías ver al menos un «hallazgo» sin corregir; Bonferroni nunca da más."
print(f"✅ Reto 1 superado: media {r_cruda['media']:.1f} → {r_ok['media']:.2f} NTU sin errores (g1 = {r_ok['asimetria']:.2f}) | Tukey {n_at_turb} → {n_at_log} puntos en log | λ̂ = {lam_hat:.2f}, IC [{ic_lambda[0]:.2f}, {ic_lambda[1]:.2f}] | ruido: {n_raw} «hallazgos» sin corregir (esperados ≈ {0.05 * 190:.1f}), {n_bonf} con Bonferroni")

---
## Reto 2: Nitratos que faltan — diagnóstico, sesgo y reglas de Rubin 🕳️

**Contexto:** `datos` trae `conductividad`, `dureza` y `nitratos` de 1 000 muestras de agua **completas**, así que conoces el parámetro verdadero. Se simula que **30 %** de los `nitratos` se pierde por tres mecanismos distintos (`faltas["MCAR"]`, `faltas["MAR"]` y `faltas["MNAR"]`; `True` = dato perdido). En la vida real **no** conocerías el valor perdido: aquí lo conoces para medir el daño. Las variables observables son `X_OBS = ["conductividad", "dureza"]`.

**Enunciado**
1. `mu_real`: media de `nitratos` en los datos completos.
2. `auc_falta(df, falta)`: AUC de predecir el indicador de falta con una regresión logística sobre `X_OBS`, evaluada con `cross_val_predict(..., cv=5, method="predict_proba")`. Guarda `auc = {mecanismo: AUC}`.
3. `sesgo_cc(df, falta)`: media de los nitratos **observados** menos `mu_real`. `sesgo_reg(df, falta)`: media tras **imputar** los faltantes con la predicción de una regresión lineal (`LinearRegression` ajustada solo con lo observado) menos `mu_real`. Guarda `sesgo = {"cc": {mec: ...}, "reg": {mec: ...}}`.
4. Con el mecanismo **MAR**: `ratio_sd_media` = desviación estándar de `nitratos` tras imputar con la **media observada** dividida por la desviación real, y `ratio_sd_est` = lo mismo tras `imputar_estocastica(datos, faltas["MAR"], np.random.default_rng(5))`.
5. **Reglas de Rubin:** implementa `rubin(est, var)` → diccionario con `q` (promedio de las estimaciones), `U` (promedio de las varianzas), `B` (varianza **entre** imputaciones, divisor $m-1$) y `T` $=U+(1+1/m)\,B$. Genera `M = 20` imputaciones estocásticas bajo MAR (semillas `np.random.default_rng(100 + j)`); de cada una toma la **media** (`est`) y la **varianza de la media** $s^2/n$ (`var`). Obtén `pool = rubin(est, var)` e `ic_mi` $=q\pm1.96\sqrt{T}$.

**Criterios de aceptación:** el AUC bajo MCAR es cercano a 0.5 y bajo MAR es alto; borrar casos sesga bajo MAR pero la regresión lo corrige; **bajo MNAR la regresión no basta**; la media encoge la desviación y la imputación estocástica la conserva; y `T > U` (la imputación única subestima la incertidumbre).

**Puntaje sugerido:** 20 puntos.

In [ ]:
X_OBS = ["conductividad", "dureza"]

# TODO: completa
mu_real = None


def auc_falta(df, falta):
    # TODO: AUC de predecir `falta` desde X_OBS (regresión logística, validación cruzada de 5 pliegues)
    return None


auc = {}          # {"MCAR": ..., "MAR": ..., "MNAR": ...}


def sesgo_cc(df, falta):
    # TODO: media de los nitratos observados - mu_real
    return None


def sesgo_reg(df, falta):
    # TODO: media tras imputar con LinearRegression (ajustada solo con lo observado) - mu_real
    return None


sesgo = {"cc": {}, "reg": {}}     # {"cc": {mec: sesgo}, "reg": {mec: sesgo}}
ratio_sd_media = ratio_sd_est = None


def rubin(est, var):
    # TODO: {"q": ..., "U": ..., "B": ..., "T": ...}
    return None


M = 20
est = var = None        # listas de M medias y M varianzas de la media (una por imputación)
pool = ic_mi = None

In [ ]:
# Autoverificación del Reto 2 (no modificar)
assert mu_real is not None and abs(mu_real - datos["nitratos"].mean()) < 1e-12, "mu_real = datos['nitratos'].mean() (los datos completos)."
assert set(auc) == set(MECANISMOS), "auc debe tener una clave por mecanismo (MCAR, MAR, MNAR)."
_p = {m: cross_val_predict(LogisticRegression(max_iter=1000), datos[X_OBS].to_numpy(), faltas[m], cv=5, method="predict_proba")[:, 1] for m in MECANISMOS}
assert all(abs(auc[m] - roc_auc_score(faltas[m], _p[m])) < 1e-6 for m in MECANISMOS), "auc[m] = roc_auc_score(falta, probabilidades de cross_val_predict con cv=5)."
assert auc["MCAR"] < 0.62 and auc["MAR"] > 0.85, "Bajo MCAR la falta no depende de lo observado (AUC ≈ 0.5); bajo MAR sí (AUC alto)."
assert auc["MNAR"] > 0.7, "Aunque el MNAR depende del propio nitrato, con lo observado se ve como un MAR (AUC alto): el diagnóstico NO puede distinguirlos."
for _m in MECANISMOS:
    _o = ~faltas[_m]
    _cc = datos.loc[_o, "nitratos"].mean() - mu_real
    _lr = LinearRegression().fit(datos.loc[_o, X_OBS], datos.loc[_o, "nitratos"])
    _y = datos["nitratos"].copy(); _y[~_o] = _lr.predict(datos.loc[~_o, X_OBS])
    assert abs(sesgo["cc"][_m] - _cc) < 1e-9 and abs(sesgo["reg"][_m] - (_y.mean() - mu_real)) < 1e-9, f"Revisa los sesgos de {_m}: cc = media observada - mu_real; reg = media de la serie completada con la regresión - mu_real."
assert abs(sesgo["cc"]["MCAR"]) < 0.4, "Bajo MCAR borrar filas casi no sesga la media."
assert sesgo["cc"]["MAR"] < -0.8 and abs(sesgo["reg"]["MAR"]) < 0.3, "Bajo MAR borrar casos subestima la media (faltan los nitratos altos), pero la regresión sobre lo observado lo corrige."
assert abs(sesgo["reg"]["MNAR"]) > 0.25 and abs(sesgo["reg"]["MNAR"]) > abs(sesgo["reg"]["MAR"]), "Bajo MNAR la regresión NO corrige el sesgo: lo que hace falta no se explica con lo observado."
_o = ~faltas["MAR"]; _sd = datos["nitratos"].std()
_ym = datos["nitratos"].where(_o, datos.loc[_o, "nitratos"].mean())
assert abs(ratio_sd_media - _ym.std() / _sd) < 1e-9 and ratio_sd_media < 0.85, "Imputar con la media encoge la desviación (ratio < 0.85 con 30 % de faltantes)."
_ye = imputar_estocastica(datos, faltas["MAR"], np.random.default_rng(5))
assert abs(ratio_sd_est - np.std(_ye, ddof=1) / _sd) < 1e-9 and 0.9 < ratio_sd_est < 1.1, "La imputación estocástica conserva la desviación estándar (ratio ≈ 1)."
_t = rubin([1.0, 2.0, 3.0], [0.1, 0.1, 0.1])
assert _t is not None and abs(_t["q"] - 2) < 1e-12 and abs(_t["U"] - 0.1) < 1e-12 and abs(_t["B"] - 1) < 1e-12 and abs(_t["T"] - (0.1 + 4 / 3)) < 1e-12, "rubin([1,2,3], [.1,.1,.1]) debe dar q=2, U=0.1, B=1 y T = U + (1 + 1/3) · B."
assert est is not None and len(est) == len(var) == M, "est y var deben tener M = 20 elementos."
_est = [imputar_estocastica(datos, faltas["MAR"], np.random.default_rng(100 + j)) for j in range(M)]
assert np.allclose(est, [np.mean(v) for v in _est]) and np.allclose(var, [np.var(v, ddof=1) / len(v) for v in _est]), "Por imputación: est = media de la serie completada y var = varianza muestral (ddof=1) / n."
_pl = rubin(est, var)
assert all(abs(pool[k] - _pl[k]) < 1e-12 for k in "qUBT") and pool["T"] > pool["U"] > 0 and pool["B"] > 0, "T debe ser mayor que U: la imputación única subestima la incertidumbre (falta la varianza entre imputaciones B)."
assert abs(ic_mi[0] - (pool["q"] - 1.96 * math.sqrt(pool["T"]))) < 1e-9 and abs(ic_mi[1] - (pool["q"] + 1.96 * math.sqrt(pool["T"]))) < 1e-9, "ic_mi = q ± 1.96 · sqrt(T)."
assert abs(pool["q"] - mu_real) < 4 * math.sqrt(pool["T"]), "La estimación combinada debería quedar cerca de la media real (a menos de 4 errores estándar)."
print(f"✅ Reto 2 superado: AUC {({m: round(float(auc[m]), 2) for m in MECANISMOS})} | sesgo con borrado {({m: round(float(sesgo['cc'][m]), 2) for m in MECANISMOS})} | con regresión {({m: round(float(sesgo['reg'][m]), 2) for m in MECANISMOS})} | sd/sd_real: media {ratio_sd_media:.2f}, estocástica {ratio_sd_est:.2f} | Rubin: q = {pool['q']:.2f}, √T = {math.sqrt(pool['T']):.3f} (√U = {math.sqrt(pool['U']):.3f})")

---
## Reto 3: Registros de campo sucios y un recibo que cierra 🧽

**Contexto:** los operarios del acueducto anotan a mano una muestra por fila (`encuesta`: `id_muestra`, `vereda`, `fecha`, `turbidez_ntu`, `volumen`, `unidad`). Los registros llegan con defectos sembrados: la vereda escrita de mil formas (mayúsculas, espacios, sin tildes, erratas), **8 veredas que no están del catálogo** `VEREDAS`, **centinelas** (`-999` = «sin lectura», `9999` = «sensor saturado») en la turbidez, volúmenes en **mL o L** según `unidad`, fechas como texto `dd/mm/aaaa` y **filas duplicadas**. No puedes perder ni inventar filas.

**Enunciado** (trabaja sobre `df = encuesta.copy()`)
1. `normalizar(texto)`: minúsculas, sin tildes (`unicodedata.normalize("NFKD", ...)` y quitar marcas combinantes) y espacios internos colapsados a uno solo, sin espacios en los extremos.
2. `canonizar(texto)`: nombre del catálogo más parecido a `normalizar(texto)` con `difflib.get_close_matches(..., n=1, cutoff=0.8)`; devuelve `None` si ninguno alcanza el corte. Crea `df["vereda_canon"]`.
3. `df["turbidez"]`: `turbidez_ntu` con los centinelas convertidos en `NaN`; `n_centinelas` = cuántos eran centinelas.
4. `df["volumen_l"]`: volumen en **litros**. `df["fecha_dt"]`: fecha leída con **formato explícito** `%d/%m/%Y`. `n_mal_leidas`: cuántas fechas se leerían **distinto** (y sin error) con el formato mes-primero `%m/%d/%Y`.
5. `df["motivo"]`: `"duplicado"` (fila **idéntica** a una anterior en las columnas **originales**; se conserva la primera aparición), si no `"vereda_desconocida"` (sin vereda canónica), si no `None`. `limpio` = filas con `motivo` nulo; `cuarentena` = el resto; `recibo = {"entran": ..., "limpio": ..., "cuarentena": ...}`.

**Criterios de aceptación:** `entran = limpio + cuarentena` (sin perder filas); cada vereda canónica coincide con la verdad (y las 8 desconocidas quedan en cuarentena, no «forzadas» al catálogo más cercano); los centinelas no quedan como números; la unidad se convierte bien; y `limpio` no repite `id_muestra`.

**Puntaje sugerido:** 20 puntos.

In [ ]:
df = encuesta.copy()


def normalizar(texto):
    # TODO: minúsculas, sin tildes y espacios colapsados
    return None


def canonizar(texto):
    # TODO: nombre del catálogo más parecido (difflib, cutoff=0.8) o None
    return None


# TODO: completa
# df["vereda_canon"] = ...
# df["turbidez"] = ...        # centinelas -999 y 9999 -> NaN
n_centinelas = None
# df["volumen_l"] = ...
# df["fecha_dt"] = ...        # formato explícito %d/%m/%Y
n_mal_leidas = None         # fechas que cambian si se lee mes-primero (%m/%d/%Y) sin error
# df["motivo"] = ...          # "duplicado" > "vereda_desconocida" > None

limpio = cuarentena = recibo = None

In [ ]:
# Autoverificación del Reto 3 (no modificar)
assert normalizar("  CAÑADA   Larga ") == "canada larga" and normalizar("Pozo  Azul") == "pozo azul", "normalizar: minúsculas, sin tildes (NFKD y quitar marcas combinantes) y espacios colapsados (' '.join(texto.split()))."
assert canonizar("Sbaneta") == "Sabaneta" and canonizar("QUEBRADA  honda") == "Quebrada Honda" and canonizar("Las Palmeras") is None, "canonizar: get_close_matches sobre los nombres normalizados del catálogo (cutoff=0.8) y devolver el nombre ORIGINAL del catálogo; None si no hay candidato."
for _c in ("vereda_canon", "turbidez", "volumen_l", "fecha_dt", "motivo"):
    assert _c in df.columns, f"Falta la columna df['{_c}']."
_ver = _VERDAD["vereda"]
_igual = (df["vereda_canon"].isna() & _ver.isna()) | (df["vereda_canon"] == _ver)
assert bool(_igual.all()), f"{int((~_igual).sum())} filas con vereda_canon incorrecta: cada vereda sucia debe ir a SU nombre del catálogo y las que no existen deben quedar en None (no las fuerces al más cercano)."
_cent = encuesta["turbidez_ntu"].isin([-999, 9999])
assert n_centinelas == int(_cent.sum()) >= 10, "n_centinelas = filas con turbidez_ntu en {-999, 9999}."
assert bool(df.loc[_cent, "turbidez"].isna().all()) and bool(np.allclose(df.loc[~_cent, "turbidez"], encuesta.loc[~_cent, "turbidez_ntu"])), "Los centinelas deben quedar como NaN y el resto de la turbidez no cambia."
assert bool(np.allclose(df["volumen_l"], _VERDAD["volumen_l"])), "volumen_l: divide entre 1000 los registros con unidad 'mL' y deja los de 'L'."
assert bool((df["fecha_dt"] == _VERDAD["fecha"]).all()), "fecha_dt = pd.to_datetime(fecha, format='%d/%m/%Y')."
_mp = pd.to_datetime(encuesta["fecha"], format="%m/%d/%Y", errors="coerce")
assert n_mal_leidas == int((_mp.notna() & (_mp != _VERDAD["fecha"])).sum()) and n_mal_leidas > 0, "n_mal_leidas: fechas que el formato mes-primero lee sin error pero con otro valor (día ≤ 12 y día ≠ mes)."
_dup = encuesta.duplicated()
assert int(_dup.sum()) == 10 and (df["motivo"] == "duplicado").sum() == int(_dup.sum()), "Hay 10 filas duplicadas (idénticas a una anterior): motivo 'duplicado' para ellas."
assert (df.loc[~_dup & _ver.isna(), "motivo"] == "vereda_desconocida").all() and df.loc[~_dup & _ver.notna(), "motivo"].isna().all(), "Entre las no duplicadas, 'vereda_desconocida' si no hay vereda canónica y None si todo está bien."
assert limpio is not None and cuarentena is not None and len(limpio) == int(df["motivo"].isna().sum()) and len(cuarentena) == int(df["motivo"].notna().sum()), "limpio = filas con motivo nulo; cuarentena = filas con motivo."
assert recibo == {"entran": len(encuesta), "limpio": len(limpio), "cuarentena": len(cuarentena)} and recibo["entran"] == recibo["limpio"] + recibo["cuarentena"], "El recibo debe cumplir entran = limpio + cuarentena."
assert limpio["id_muestra"].is_unique and limpio["vereda_canon"].isin(VEREDAS).all(), "limpio no puede repetir id_muestra y todas sus veredas deben estar en el catálogo."
print(f"✅ Reto 3 superado: recibo {recibo} | motivos {cuarentena['motivo'].value_counts().to_dict()} | {n_centinelas} filas con centinela | {n_mal_leidas} fechas se habrían leído mal en silencio con mes-primero")

---
## Reto 4: PCA sin trampas — escala, número de componentes y fuga 🧭

**Contexto:** una red de 10 sensores fisicoquímicos (`X_sens`, 300 × 10, columnas en `COLUMNAS`) se mueve por `N_FACTORES = 3` causas latentes, pero cada sensor mide en una **escala** muy distinta (la conductividad varía cientos de unidades y el pH una o dos). Quieres comprimir la red y, aparte, advertir a un colega que «elegir las mejores variables y *luego* validar» es una trampa.

**Enunciado**
1. `pca_svd(X, estandarizar)` → `(lam, comp, scores)`: centra `X` (y, si `estandarizar`, divide cada columna por su desviación **muestral** $n-1$), aplica la **SVD** y devuelve los autovalores $\lambda_j=s_j^2/(n-1)$ en orden decreciente, los componentes (filas) y las puntuaciones $Z\,V$.
2. `prop_sin` y `prop_con`: proporción de la varianza que explica el **primer** componente sin y con estandarizar; `var_dom`: nombre de la columna con mayor peso absoluto en el primer componente **sin** estandarizar.
3. Con los datos estandarizados: `k_95` (menor $k$ con varianza acumulada $\ge 0.95$), `k_kaiser` (cuántos $\lambda>1$) y `k_paralelo` (**análisis paralelo**: compara cada $\lambda_j$ con su percentil 95 en 100 matrices de correlación donde **cada columna se permutó** por separado; usa `np.random.default_rng(0)`).
4. `err_recon`: $\lVert Z-\hat Z_k\rVert_F^2$ al reconstruir con `k = k_paralelo` componentes, y `err_teorico` $=(n-1)\sum_{j>k}\lambda_j$.
5. **Fuga:** `Xr` (100 × 2 000) y `yr` son **ruido puro** (no hay relación). Con `cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=0)` y regresión logística, calcula `acc_fuga` (eligiendo las 20 mejores variables con `SelectKBest(f_classif, k=20)` sobre **todos** los datos y *luego* validando) y `acc_pipe` (la selección **dentro** de un `make_pipeline`, es decir, solo con cada pliego de entrenamiento).

**Criterios de aceptación:** tu PCA coincide con `sklearn.decomposition.PCA`; sin estandarizar domina la variable de mayor escala; el análisis paralelo recupera los 3 factores; la identidad del error de reconstrucción se cumple; y `acc_fuga` es engañosamente alta frente a `acc_pipe`, que ronda el azar.

**Puntaje sugerido:** 25 puntos.

In [ ]:
def pca_svd(X, estandarizar):
    # TODO: centrar (y estandarizar con ddof=1 si se pide), SVD, y devolver (lam, comp, scores)
    return None, None, None


# TODO: completa
lam_sin = lam_con = None
prop_sin = prop_con = None
var_dom = None                   # nombre de la columna dominante en PC1 SIN estandarizar

k_95 = k_kaiser = k_paralelo = None
err_recon = err_teorico = None

rng_r = np.random.default_rng(SEMILLA + 5)
Xr = rng_r.normal(size=(100, 2000))
yr = rng_r.integers(0, 2, 100)
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=0)
acc_fuga = acc_pipe = None

In [ ]:
# Autoverificación del Reto 4 (no modificar)
for _est in (False, True):
    _lam, _comp, _sc = pca_svd(X_sens, _est)
    assert _lam is not None, "pca_svd debe devolver (lam, comp, scores)."
    _Z = X_sens - X_sens.mean(0)
    if _est:
        _Z = _Z / X_sens.std(0, ddof=1)
    _sk = PCA().fit(_Z)
    assert np.allclose(_lam, _sk.explained_variance_), "Los autovalores deben ser s**2 / (n - 1) (coinciden con explained_variance_ de scikit-learn)."
    assert np.allclose(np.abs(_comp), np.abs(_sk.components_), atol=1e-8), "Los componentes (filas de Vt) coinciden con scikit-learn salvo el signo."
    assert np.allclose(np.abs(_sc), np.abs(_sk.transform(_Z)), atol=1e-6), "Las puntuaciones son Z @ comp.T (coinciden con transform salvo el signo)."
assert abs(np.sum(pca_svd(X_sens, True)[0]) - 10) < 1e-8, "Al estandarizar (ddof=1) la suma de los autovalores es p = 10 (traza de la matriz de correlación)."
_ls, _lc = pca_svd(X_sens, False)[0], pca_svd(X_sens, True)[0]
assert abs(prop_sin - _ls[0] / _ls.sum()) < 1e-9 and abs(prop_con - _lc[0] / _lc.sum()) < 1e-9, "prop = autovalor_1 / suma de autovalores."
assert prop_sin > 0.8 and prop_con < 0.6, "Sin estandarizar un solo sensor (el de mayor escala) domina PC1; estandarizado, PC1 explica mucho menos."
assert var_dom == COLUMNAS[int(np.argmax(X_sens.var(0)))] == "conductividad", "var_dom: la columna con mayor |peso| en el primer componente sin estandarizar (la de mayor varianza)."
_acum = np.cumsum(_lc) / _lc.sum()
assert k_95 == int(np.argmax(_acum >= 0.95)) + 1 and k_kaiser == int((_lc > 1).sum()), "k_95: primer k con varianza acumulada >= 0.95; k_kaiser: cuántos autovalores > 1 (de la matriz de correlación)."
assert k_paralelo == N_FACTORES and k_95 > k_paralelo, "El análisis paralelo debe recuperar los 3 factores sembrados; el criterio del 95 % pide muchos más componentes (hay ruido en cada sensor)."
_Z = (X_sens - X_sens.mean(0)) / X_sens.std(0, ddof=1)
_V = np.linalg.svd(_Z, full_matrices=False)[2][:k_paralelo]
_Zh = (_Z @ _V.T) @ _V
assert abs(err_recon - np.sum((_Z - _Zh) ** 2)) < 1e-6 * err_recon, "err_recon = suma de cuadrados de (Z - reconstrucción con k componentes)."
assert abs(err_teorico - (len(_Z) - 1) * np.sum(_lc[k_paralelo:])) < 1e-6 * err_teorico and abs(err_recon - err_teorico) < 1e-6 * err_teorico, "La identidad: error de reconstrucción = (n - 1) · suma de los autovalores descartados."
_a1 = cross_val_score(LogisticRegression(max_iter=1000), SelectKBest(f_classif, k=20).fit_transform(Xr, yr), yr, cv=cv).mean()
_a2 = cross_val_score(make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000)), Xr, yr, cv=cv).mean()
assert acc_fuga is not None and acc_pipe is not None and abs(acc_fuga - _a1) < 1e-9 and abs(acc_pipe - _a2) < 1e-9, "acc_fuga: SelectKBest.fit_transform(Xr, yr) con TODOS los datos y luego cross_val_score; acc_pipe: make_pipeline(SelectKBest, LogisticRegression) con cross_val_score sobre Xr."
assert acc_fuga - acc_pipe > 0.15 and acc_pipe < 0.7, "Sobre ruido puro la exactitud honesta ronda 0.5; la inflada por la fuga debe superarla por mucho."
print(f"✅ Reto 4 superado: PC1 {prop_sin:.1%} sin estandarizar (domina {var_dom}) frente a {prop_con:.1%} estandarizado | k: 95 % → {k_95}, Kaiser → {k_kaiser}, paralelo → {k_paralelo} | error de reconstrucción {err_recon:.1f} = {err_teorico:.1f} | ruido puro: exactitud {acc_fuga:.2f} con fuga frente a {acc_pipe:.2f} dentro del pipeline")

---
## Reto 5: Conclusiones 📝

**Puntaje sugerido:** 15 puntos (sin código). Escribe **5 a 8 líneas** en la celda de abajo interpretando *tus* resultados. Debe responder:

- ¿Por qué Tukey marcó **menos** puntos sobre $\ln x$ que sobre $x$, y qué te dice el IC de $\lambda$ frente a «no transformar»? ¿Cuántos «hallazgos» salieron en ruido puro, cuántos esperarías por azar y qué hace Bonferroni (y qué cuesta)?
- Con tus AUC y tus sesgos: ¿por qué el MNAR **se parece** al MAR con lo observado, y por qué entonces la regresión no lo corrige? ¿Qué aporta `B` en las reglas de Rubin que la imputación única ignora?
- ¿Por qué mostrar `n_mal_leidas` justifica el formato explícito, y por qué el recibo (`entran = limpio + cuarentena`) es mejor que descartar filas «raras»?
- ¿Por qué estandarizar cambió tan radicalmente PC1, y por qué el análisis paralelo y el criterio del 95 % difieren? Con tu `acc_fuga` y `acc_pipe`, ¿qué conclusión errónea habría salido de la fuga? Y, a diferencia de PCA, ¿por qué no interpretarías las **distancias entre grupos** de un gráfico t-SNE?

---
#### ✍️ Tus conclusiones

_Escribe aquí tus 5-8 líneas. Reemplaza este texto por tu análisis._

---
### 🧠 Rúbrica de evaluación

| Criterio | Puntos |
|---|---|
| Reto 1: resumen con IC, error de captura, Tukey, Box-Cox con su IC y comparaciones múltiples | 20 |
| Reto 2: diagnóstico de la falta, sesgo con verdad conocida, imputación estocástica y reglas de Rubin | 20 |
| Reto 3: normalización de texto, centinelas, unidades, fechas explícitas, cuarentena y conservación de filas | 20 |
| Reto 4: PCA con la SVD, escala, número de componentes, identidad de reconstrucción y fuga | 25 |
| Reto 5: conclusiones argumentadas con tus cifras (5-8 líneas) | 15 |
| **Total** | **100** |

### ✅ Checklist de entrega

- [ ] Reinicié el kernel y ejecuté *Run All* sin errores.
- [ ] No quedan `# TODO` ni variables en `None`.
- [ ] Las cuatro celdas de autoverificación imprimen ✅.
- [ ] Ninguna transformación se ajustó con datos que luego uso para validar (sin fugas).
- [ ] Mi recibo de limpieza cumple *entran = limpio + cuarentena*.
- [ ] Escribí mis conclusiones con cifras de mis propios resultados.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Asignatura: Estadística, Análisis y Representación de Datos</i>
  </p>
</div>